# Online Retail Sales: Data Cleaning & Analysis

Cleans the UCI Online Retail transaction data, then analyzes revenue by month, product, customer and country.

**Input:** `online_retail.csv`  **Output:** `online_retail_cleaned.csv` and four chart images (PNG).
Run all cells from top to bottom.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Folder the notebook is running in (the CSV and the output files live here)
BASE_DIR = os.getcwd()

df = pd.read_csv(os.path.join(BASE_DIR, 'online_retail.csv'), encoding='ISO-8859-1')
print("Original shape:", df.shape)

## 1. Data cleaning

In [ ]:
# Step 1: drop rows with missing CustomerID (sales can't be attributed to a customer)
before = df.shape[0]
df = df.dropna(subset=['CustomerID'])
print(f"Removed {before - df.shape[0]} rows with missing CustomerID -> {df.shape}")

In [ ]:
# Step 2: remove duplicate rows
before = df.shape[0]
df = df.drop_duplicates()
print(f"Removed {before - df.shape[0]} duplicate rows")

In [ ]:
# Step 3: separate cancelled orders (invoice numbers starting with 'C')
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
cancelled = df[df['InvoiceNo'].str.startswith('C')]
df = df[~df['InvoiceNo'].str.startswith('C')]
print(f"Separated {cancelled.shape[0]} cancelled order rows")

In [ ]:
# Step 4: remove invalid rows (quantity or price <= 0)
before = df.shape[0]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
print(f"Removed {before - df.shape[0]} rows with invalid quantity/price")

In [ ]:
# Step 5: remove original orders that were later cancelled.
# Removing only the cancellation rows (step 3) would leave the original order in the data
# and overstate revenue. Each cancellation is matched one-to-one with an original order
# for the same customer, product and quantity.
keys = ['CustomerID', 'StockCode', 'Quantity']

cancel_counts = (
    cancelled.assign(Quantity=-cancelled['Quantity'])
    .groupby(keys).size().reset_index(name='n_cancelled')
)

df = df.reset_index(drop=True)
df['_nth'] = df.groupby(keys).cumcount()
df = df.merge(cancel_counts, on=keys, how='left')
df['n_cancelled'] = df['n_cancelled'].fillna(0)

reversed_mask = df['_nth'] < df['n_cancelled']
print(f"Removed {reversed_mask.sum()} original order rows that were later cancelled")

df = df[~reversed_mask].drop(columns=['_nth', 'n_cancelled']).reset_index(drop=True)

In [ ]:
# Steps 6-8: standardize text, convert dates, create the Revenue column
df['Description'] = df['Description'].str.strip().str.upper()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue'] = df['Quantity'] * df['UnitPrice']

In [ ]:
# Final check and save
print("Final shape:", df.shape)
print(df.isnull().sum())
print(df.head())

df.to_csv(os.path.join(BASE_DIR, 'online_retail_cleaned.csv'), index=False)

## 2. Analysis

In [ ]:
# Monthly revenue
df['Month'] = df['InvoiceDate'].dt.to_period('M')
monthly_revenue = df.groupby('Month')['Revenue'].sum().reset_index()
print(monthly_revenue)
print(f"\nTotal revenue: {df['Revenue'].sum():,.2f}")

In [ ]:
# Top 10 products by revenue.
# Fees and adjustments (postage, manual entries, etc.) are not real products, so they are excluded here.
non_products = ['POSTAGE', 'DOTCOM POSTAGE', 'MANUAL', 'CARRIAGE', 'BANK CHARGES',
                'AMAZON FEE', 'SAMPLES', 'CRUK COMMISSION', 'DISCOUNT']

top_products = (
    df[~df['Description'].isin(non_products)]
    .groupby('Description')['Revenue'].sum()
    .sort_values(ascending=False).head(10)
)
print(top_products)

In [ ]:
# Top 10 customers by revenue
top_customers = df.groupby('CustomerID')['Revenue'].sum().sort_values(ascending=False).head(10)
print(top_customers)
print(f"\nTop 10 customers share of total revenue: {top_customers.sum() / df['Revenue'].sum():.1%}")

In [ ]:
# Revenue by country
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).reset_index()
print(country_revenue)
print(f"\nUnited Kingdom share of total revenue: "
      f"{country_revenue.loc[country_revenue['Country'] == 'United Kingdom', 'Revenue'].iloc[0] / df['Revenue'].sum():.1%}")

In [ ]:
# Average order value (revenue per invoice, not per line item)
order_totals = df.groupby('InvoiceNo')['Revenue'].sum()
print(f"Number of orders: {order_totals.shape[0]}")
print(f"Average order value: {order_totals.mean():.2f}")
print(f"Median order value: {order_totals.median():.2f}")

## 3. Charts

Each chart is also saved as a PNG next to the notebook.

In [ ]:
# Monthly revenue trend
monthly_plot = monthly_revenue.copy()
monthly_plot['Month'] = monthly_plot['Month'].astype(str)

plt.figure(figsize=(10, 6))
sns.lineplot(x='Month', y='Revenue', data=monthly_plot, marker='o', color='#4C72B0')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Total Revenue (£)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'sales_monthly_trend.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 products
plt.figure(figsize=(12, 8))
sns.barplot(x='Revenue', y='Description', data=top_products.reset_index(), orient='h', color='#4C72B0')
plt.title('Top 10 Best-Selling Products by Revenue')
plt.xlabel('Total Revenue (£)')
plt.ylabel('Product Description')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'sales_top_products.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 customers
plt.figure(figsize=(12, 8))
sns.barplot(x='Revenue', y=top_customers.reset_index()['CustomerID'].astype(int).astype(str),
            data=top_customers.reset_index(), orient='h', color='#4C72B0')
plt.title('Top 10 Customers by Spend')
plt.xlabel('Total Revenue (£)')
plt.ylabel('Customer ID')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'sales_top_customers.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Revenue by country
plt.figure(figsize=(12, 8))
sns.barplot(x='Revenue', y='Country', data=country_revenue, orient='h', color='#4C72B0')
plt.title('Revenue by Country')
plt.xlabel('Total Revenue (£)')
plt.ylabel('Country')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'revenue-by-country.png'), dpi=150, bbox_inches='tight')
plt.show()